# 06 — Modelo final

## Objetivo

Como congelar a decisão e avaliar o teste uma única vez?

O modelo final é o Gradient Boosting otimizado, sem pesos ou SMOTENC, com
limiar 0,27. Para preservar a comparação técnica, ele permanece treinado apenas
no conjunto de treino.

In [ ]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn

from src.auxiliares import (
    ALVO,
    COLUNAS_NOMINAIS,
    COLUNAS_STATUS,
    PARAMETROS_REFERENCIA,
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_gradiente,
    separar_dados,
)
from src.visual_utils import grafico_matriz_confusao

LIMIAR_FINAL = 0.27
dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)
caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
parametros = json.loads(caminho_parametros.read_text()) if caminho_parametros.exists() else PARAMETROS_REFERENCIA.copy()

## O desempenho de validação continua coerente?

## Quais decisões ficam congeladas antes do teste?

In [ ]:
decisoes = pd.Series({
    "modelo": "GradientBoostingClassifier otimizado",
    "parametros": parametros,
    "balanceamento": "dados originais",
    "limiar": LIMIAR_FINAL,
    "metrica_selecao": "Average Precision (AP / PR-AUC)",
    "refit_com_validacao": False,
})
decisoes.to_frame("decisao")

## Qual é o resultado no teste protegido?

In [ ]:
previsoes_teste = (probabilidades_teste >= LIMIAR_FINAL).astype(int)
fig = grafico_matriz_confusao(y_teste, previsoes_teste)
fig.show()

## Como salvar o pipeline para a aplicação?

In [ ]:
colunas_features = X_treino.columns.tolist()
campos_com_opcoes = [*COLUNAS_NOMINAIS, *COLUNAS_STATUS]
opcoes_categoricas = {
    coluna: sorted(int(valor) for valor in X_treino[coluna].unique())
    for coluna in campos_com_opcoes
}
valores_padrao = {
    coluna: (
        int(X_treino[coluna].mode().iloc[0])
        if coluna in campos_com_opcoes or coluna == "idade"
        else float(X_treino[coluna].median())
    )
    for coluna in colunas_features
}
versoes = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

In [ ]:
def valor_json(valor):
    if isinstance(valor, dict):
        return {chave: valor_json(item) for chave, item in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [valor_json(item) for item in valor]
    if isinstance(valor, np.generic):
        return valor.item()
    if isinstance(valor, float) and not np.isfinite(valor):
        return None
    return valor

matriz_teste = [
    [resultado_teste["vn"], resultado_teste["fp"]],
    [resultado_teste["fn"], resultado_teste["vp"]],
]
metricas_modelo = {
    "modelo": "Gradient Boosting otimizado",
    "limiar": LIMIAR_FINAL,
    "validacao": resultado_validacao,
    "teste": resultado_teste,
    "matriz_confusao_teste": matriz_teste,
    "observacao": "Modelo treinado somente no treino; teste avaliado uma vez.",
}
caminho_metricas = RAIZ / "models" / "metricas_modelo.json"
caminho_metricas.write_text(
    json.dumps(valor_json(metricas_modelo), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Métricas salvas em: {caminho_metricas}")

## O artefato restaurado produz a mesma probabilidade?

In [ ]:
modelo_restaurado = joblib.load(caminho_modelo)
amostra = X_validacao.iloc[:5]
probabilidades_antes = modelo_final.predict_proba(amostra)[:, 1]
probabilidades_depois = modelo_restaurado["pipeline"].predict_proba(amostra)[:, 1]
assert np.allclose(probabilidades_antes, probabilidades_depois)

pd.DataFrame({
    "probabilidade": probabilidades_depois,
    "limiar": LIMIAR_FINAL,
    "classe": (probabilidades_depois >= LIMIAR_FINAL).astype(int),
})

## Resultado

O modelo, o limiar, os valores padrão e as métricas foram serializados para o
Streamlit. O teste só foi consultado depois do congelamento das decisões.